# Operational Effort Coefficients

This notebook prepares the operational effort coefficients for the optimization model.

Since 4-car / 8-car train-set configurations are not available or verifiable for all metro lines, we do not model different train-set types. Instead, each metro line is assumed to operate with its standard train capacity.

Operational cost is represented by the number of additional standard train trips. Each additional standard train trip is assigned one unit of normalized operational effort.

In [1]:
import pandas as pd
import numpy as np

In [2]:
capacity_df = pd.read_csv("istanbul_rayli_sistem_verileri_guncel.csv")

capacity_df.head()

,Hat_Adi,Arac_Tipi,Set_Kapasitesi_Yolcu,Toplam_Arac_Sayisi,Ortalama_Hiz_km_h,Set_Uzunlugu_Vagon,Kaynak_Link
0,F1,Füniküler (Doppelmayr),375,4,30,2,https://www.metro.istanbul/Hatlarimiz
1,F2,Tarihi Tünel,170,2,10,2,https://iett.istanbul/icerik/tunel
2,F3,Füniküler,250,4,15,2,https://www.metro.istanbul/Hatlarimiz
3,F4,Füniküler,200,2,20,2,https://www.metro.istanbul/Hatlarimiz
4,M1,ABB,1000,105,30,4,https://www.metro.istanbul/Hatlarimiz


In [3]:
capacity_df.columns

Index(['Hat_Adi', 'Arac_Tipi', 'Set_Kapasitesi_Yolcu', 'Toplam_Arac_Sayisi',
       'Ortalama_Hiz_km_h', 'Set_Uzunlugu_Vagon', 'Kaynak_Link'],
      dtype='object')

In [4]:
metro_df = capacity_df[
    capacity_df["Hat_Adi"].astype(str).str.match(r"^M\d+")
].copy()

metro_df

,Hat_Adi,Arac_Tipi,Set_Kapasitesi_Yolcu,Toplam_Arac_Sayisi,Ortalama_Hiz_km_h,Set_Uzunlugu_Vagon,Kaynak_Link
4,M1,ABB,1000,105,30,4,https://www.metro.istanbul/Hatlarimiz
5,M2,Hyundai Rotem / Alstom,1600,180,40,8,https://www.metro.istanbul/Hatlarimiz
6,M3,Alstom Metropolis,1100,80,35,4,https://www.metro.istanbul/Hatlarimiz
7,M4,CAF,2200,144,38,8,https://www.metro.istanbul/Hatlarimiz
8,M5,Mitsubishi / CAF,1620,126,38,6,https://www.metro.istanbul/Hatlarimiz
9,M6,Alstom,1000,12,25,4,https://www.metro.istanbul/Hatlarimiz
10,M7,Hyundai Rotem,1100,80,38,4,https://www.metro.istanbul/Hatlarimiz
11,M8,Hyundai Rotem,1100,40,35,4,https://www.metro.istanbul/Hatlarimiz
12,M9,Alstom / Hyundai,1100,50,35,4,https://www.metro.istanbul/Hatlarimiz
13,M11,CRRC ZELC,1100,80,80,4,https://www.uab.gov.tr/


In [5]:
metro_df = metro_df[
    [
        "Hat_Adi",
        "Set_Kapasitesi_Yolcu",
        "Toplam_Arac_Sayisi",
        "Ortalama_Hiz_km_h",
        "Set_Uzunlugu_Vagon"
    ]
].copy()

metro_df

,Hat_Adi,Set_Kapasitesi_Yolcu,Toplam_Arac_Sayisi,Ortalama_Hiz_km_h,Set_Uzunlugu_Vagon
4,M1,1000,105,30,4
5,M2,1600,180,40,8
6,M3,1100,80,35,4
7,M4,2200,144,38,8
8,M5,1620,126,38,6
9,M6,1000,12,25,4
10,M7,1100,80,38,4
11,M8,1100,40,35,4
12,M9,1100,50,35,4
13,M11,1100,80,80,4


In [6]:
metro_df = metro_df.rename(columns={
    "Hat_Adi": "line",
    "Set_Kapasitesi_Yolcu": "standard_capacity_passengers",
    "Toplam_Arac_Sayisi": "total_vehicle_count",
    "Ortalama_Hiz_km_h": "avg_speed_kmh",
    "Set_Uzunlugu_Vagon": "standard_wagon_count"
})

metro_df

,line,standard_capacity_passengers,total_vehicle_count,avg_speed_kmh,standard_wagon_count
4,M1,1000,105,30,4
5,M2,1600,180,40,8
6,M3,1100,80,35,4
7,M4,2200,144,38,8
8,M5,1620,126,38,6
9,M6,1000,12,25,4
10,M7,1100,80,38,4
11,M8,1100,40,35,4
12,M9,1100,50,35,4
13,M11,1100,80,80,4


In [7]:
numeric_columns = [
    "standard_capacity_passengers",
    "total_vehicle_count",
    "avg_speed_kmh",
    "standard_wagon_count"
]

for col in numeric_columns:
    metro_df[col] = pd.to_numeric(metro_df[col], errors="coerce")

metro_df

,line,standard_capacity_passengers,total_vehicle_count,avg_speed_kmh,standard_wagon_count
4,M1,1000,105,30,4
5,M2,1600,180,40,8
6,M3,1100,80,35,4
7,M4,2200,144,38,8
8,M5,1620,126,38,6
9,M6,1000,12,25,4
10,M7,1100,80,38,4
11,M8,1100,40,35,4
12,M9,1100,50,35,4
13,M11,1100,80,80,4


In [8]:
metro_df["effort_per_extra_trip"] = 1.00
metro_df["effort_unit"] = "one additional standard train trip"
metro_df["cost_interpretation"] = "normalized operational effort, not monetary cost"

metro_df

,line,standard_capacity_passengers,total_vehicle_count,avg_speed_kmh,standard_wagon_count,effort_per_extra_trip,effort_unit,cost_interpretation
4,M1,1000,105,30,4,1.0,one additional standard train trip,"normalized operational effort, not monetary cost"
5,M2,1600,180,40,8,1.0,one additional standard train trip,"normalized operational effort, not monetary cost"
6,M3,1100,80,35,4,1.0,one additional standard train trip,"normalized operational effort, not monetary cost"
7,M4,2200,144,38,8,1.0,one additional standard train trip,"normalized operational effort, not monetary cost"
8,M5,1620,126,38,6,1.0,one additional standard train trip,"normalized operational effort, not monetary cost"
9,M6,1000,12,25,4,1.0,one additional standard train trip,"normalized operational effort, not monetary cost"
10,M7,1100,80,38,4,1.0,one additional standard train trip,"normalized operational effort, not monetary cost"
11,M8,1100,40,35,4,1.0,one additional standard train trip,"normalized operational effort, not monetary cost"
12,M9,1100,50,35,4,1.0,one additional standard train trip,"normalized operational effort, not monetary cost"
13,M11,1100,80,80,4,1.0,one additional standard train trip,"normalized operational effort, not monetary cost"


In [9]:
operational_effort_df = metro_df[
    [
        "line",
        "standard_capacity_passengers",
        "effort_per_extra_trip",
        "effort_unit",
        "cost_interpretation",
        "total_vehicle_count",
        "avg_speed_kmh",
        "standard_wagon_count"
    ]
].copy()

operational_effort_df

,line,standard_capacity_passengers,effort_per_extra_trip,effort_unit,cost_interpretation,total_vehicle_count,avg_speed_kmh,standard_wagon_count
4,M1,1000,1.0,one additional standard train trip,"normalized operational effort, not monetary cost",105,30,4
5,M2,1600,1.0,one additional standard train trip,"normalized operational effort, not monetary cost",180,40,8
6,M3,1100,1.0,one additional standard train trip,"normalized operational effort, not monetary cost",80,35,4
7,M4,2200,1.0,one additional standard train trip,"normalized operational effort, not monetary cost",144,38,8
8,M5,1620,1.0,one additional standard train trip,"normalized operational effort, not monetary cost",126,38,6
9,M6,1000,1.0,one additional standard train trip,"normalized operational effort, not monetary cost",12,25,4
10,M7,1100,1.0,one additional standard train trip,"normalized operational effort, not monetary cost",80,38,4
11,M8,1100,1.0,one additional standard train trip,"normalized operational effort, not monetary cost",40,35,4
12,M9,1100,1.0,one additional standard train trip,"normalized operational effort, not monetary cost",50,35,4
13,M11,1100,1.0,one additional standard train trip,"normalized operational effort, not monetary cost",80,80,4


In [10]:
operational_effort_df["line"].value_counts()

line
M1     1
M2     1
M3     1
M4     1
M5     1
M6     1
M7     1
M8     1
M9     1
M11    1
Name: count, dtype: int64

In [11]:
operational_effort_df.isna().sum()

line                            0
standard_capacity_passengers    0
effort_per_extra_trip           0
effort_unit                     0
cost_interpretation             0
total_vehicle_count             0
avg_speed_kmh                   0
standard_wagon_count            0
dtype: int64

In [12]:
operational_effort_df.to_csv(
    "operational_effort_coefficients.csv",
    index=False
)

In [13]:
check_df = pd.read_csv("operational_effort_coefficients.csv")
check_df

,line,standard_capacity_passengers,effort_per_extra_trip,effort_unit,cost_interpretation,total_vehicle_count,avg_speed_kmh,standard_wagon_count
0,M1,1000,1.0,one additional standard train trip,"normalized operational effort, not monetary cost",105,30,4
1,M2,1600,1.0,one additional standard train trip,"normalized operational effort, not monetary cost",180,40,8
2,M3,1100,1.0,one additional standard train trip,"normalized operational effort, not monetary cost",80,35,4
3,M4,2200,1.0,one additional standard train trip,"normalized operational effort, not monetary cost",144,38,8
4,M5,1620,1.0,one additional standard train trip,"normalized operational effort, not monetary cost",126,38,6
5,M6,1000,1.0,one additional standard train trip,"normalized operational effort, not monetary cost",12,25,4
6,M7,1100,1.0,one additional standard train trip,"normalized operational effort, not monetary cost",80,38,4
7,M8,1100,1.0,one additional standard train trip,"normalized operational effort, not monetary cost",40,35,4
8,M9,1100,1.0,one additional standard train trip,"normalized operational effort, not monetary cost",50,35,4
9,M11,1100,1.0,one additional standard train trip,"normalized operational effort, not monetary cost",80,80,4


## Final Assumptions

1. The model does not use 4-car / 8-car train-set alternatives.
2. Each metro line is represented with its standard train capacity.
3. The operational cost is not a monetary cost.
4. The operational cost is represented as normalized operational effort.
5. One additional standard train trip corresponds to one unit of operational effort.
6. Passenger occupancy does not directly affect operational cost. Passenger demand affects the load ratio, which is handled separately in the optimization model.